In [1]:
import prepocess, importlib
importlib.reload(prepocess)
from prepocess import preprocess
import pandas as pd


file_path = "syn_data/syn_net_p0.8_mu0.2_1.csv"
node2id, num_nodes = preprocess(file_path=file_path)

50 nodes in the link stream


/Users/acw721/Desktop/research/linkstream/prepocess.py:4: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  link_stream = pd.read_csv(


In [2]:
link_stream = pd.read_csv('preprocessed/syn_net_p0.8_mu0.2_1_pcs.csv')

In [3]:
from tgn.utils.my_data import get_data  
import importlib
import sys
importlib.reload(sys.modules['tgn.utils.my_data'])

data = 'syn_net_p0.8_mu0.2_pcs'

node_feat, edge_feat, full_data = get_data(data, "preprocessed/syn_net_p0.8_mu0.2_1_pcs.csv", node_embedding_method="one-hot")

max_idx = max(full_data.unique_nodes)

cannot find (./data/syn_net_p0.8_mu0.2_pcs.npy), use zero-vector for edge feat (dim=16)...
Use one-hot init for node embedding. 
The dataset has 469 interactions, involving 50 different nodes


In [ ]:
from tgn.utils.utils import EarlyStopMonitor, get_neighbor_finder, MLP

ngh_finder = get_neighbor_finder(full_data, uniform=True, max_node_idx=max_idx)

In [5]:
from tgn.utils.my_data import get_data, compute_time_statistics, compute_time_statistics_undirected
import importlib
import sys
importlib.reload(sys.modules['tgn.utils.my_data'])

mean_time_shift, std_time_shift= compute_time_statistics_undirected(full_data.sources, 
                                full_data.destinations, 
                                full_data.timestamps)


In [33]:
importlib.reload(sys.modules['tgn.model.my_tgn'])
from tgn.model.my_tgn import TGN

from pathlib import Path


import math
import logging
import time
import sys
import argparse
import numpy as np
import pickle
from pathlib import Path
import torch


NUM_NEIGHBORS = 20
NUM_HEADS = 4
DROP_OUT = 0.1
NUM_LAYER = 2
LEARNING_RATE = 0.001
NODE_DIM = node_feat.shape[1]
TIME_DIM = 16
USE_MEMORY = True
MESSAGE_DIM = 100
MEMORY_DIM = NODE_DIM
num_communities = 5
device = 'cuda' if torch.cuda.is_available() else 'mps'
prefix = 'syn_net'
aggregator = 'mean'
memory_update_at_end = False
embedding_module = 'graph_attention' # graph_attention, graph_sum, time, identity
message_function = 'mlp'


Path("./saved_models/").mkdir(parents=True, exist_ok=True)
Path("./saved_checkpoints/").mkdir(parents=True, exist_ok=True)
MODEL_SAVE_PATH = f'./saved_models/{prefix}-{data}.pth'
get_checkpoint_path = lambda \
    epoch: f'./saved_checkpoints/{prefix}-{data}-{epoch}.pth'
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
Path("log/").mkdir(parents=True, exist_ok=True)
fh = logging.FileHandler('log/{}.log'.format(str(time.time())))
fh.setLevel(logging.DEBUG)
ch = logging.StreamHandler()
ch.setLevel(logging.WARN)
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
fh.setFormatter(formatter)
ch.setFormatter(formatter)
logger.addHandler(fh)
logger.addHandler(ch)


tgn = TGN(
    neighbor_finder=ngh_finder,
    node_features=node_feat,
    edge_features=edge_feat,
    device=device,
    n_layers=NUM_LAYER,
    n_heads=NUM_HEADS,
    dropout=DROP_OUT,
    use_memory=USE_MEMORY,
    message_dimension=MESSAGE_DIM,
    memory_dimension=MEMORY_DIM,
    n_neighbors=NUM_NEIGHBORS,
    mean_time_shift=mean_time_shift,
    std_time_shift=std_time_shift,
    use_destination_embedding_in_message=True,
    use_source_embedding_in_message=True,
    memory_update_at_start=not memory_update_at_end,
    embedding_module_type=embedding_module,
    message_function=message_function,
    aggregator_type=aggregator, 
    num_communities=num_communities
).to(device)

optimizer = torch.optim.Adam(tgn.parameters(), lr=LEARNING_RATE)

BATCH_SIZE = 500

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
Path("log/").mkdir(parents=True, exist_ok=True)
fh = logging.FileHandler('log/{}.log'.format(str(time.time())))
fh.setLevel(logging.DEBUG)
ch = logging.StreamHandler()
ch.setLevel(logging.WARN)
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
fh.setFormatter(formatter)
ch.setFormatter(formatter)
logger.addHandler(fh)
logger.addHandler(ch)


Path("./saved_models/").mkdir(parents=True, exist_ok=True)
Path("./saved_checkpoints/").mkdir(parents=True, exist_ok=True)
MODEL_SAVE_PATH = f'./saved_models/{prefix}-{data}.pth'
get_checkpoint_path = lambda \
    epoch: f'./saved_checkpoints/{prefix}-{data}-{epoch}.pth'


num_instance = len(full_data.sources)
num_batches = math.ceil(len(full_data.sources) / BATCH_SIZE)
logger.debug(f'num_batches: {num_batches}')

DEBUG:root:num_batches: 1


In [4]:

from pathlib import Path
import math
import logging
import time
import sys
import argparse
import numpy as np
import pickle
from pathlib import Path
import torch


BATCH_SIZE = 500

num_instance = len(full_data.sources)
num_batches = math.ceil(len(full_data.sources) / BATCH_SIZE)
print(f'num_batches: {num_batches}')

num_batches: 1


In [5]:
import torch

class ExpKernelActivity:
    """
    Maintain k_u(t) = sum_{events at times t_i < t} exp(-beta*(t - t_i))
    with lazy decay.

    - query(nodes, t): returns k(nodes, t) WITHOUT modifying the stored state
    - update_endpoints(src, dst, t): applies decay-to-t and then increments by 1 for endpoints
    """
    def __init__(self, num_nodes, beta, device, dtype=torch.float32, t0=0.0):
        self.beta = float(beta)
        self.device = device
        self.k = torch.zeros(num_nodes, device=device, dtype=dtype)
        self.last_t = torch.full((num_nodes,), float(t0), device=device, dtype=dtype)

    def _decay_to(self, nodes: torch.Tensor, t: torch.Tensor):
        """
        In-place: decay k[nodes] from last_t to time t (t can be scalar or [N]).
        """
        # t shape: scalar or [N]
        last = self.last_t[nodes]
        dt = torch.clamp(t.to(last.dtype) - last, min=0.0)
        if self.beta > 0:
            self.k[nodes] = self.k[nodes] * torch.exp(-self.beta * dt)
        # update last_t to current t
        self.last_t[nodes] = t.to(last.dtype)

    @torch.no_grad()
    def update_endpoints(self, src: torch.Tensor, dst: torch.Tensor, t: torch.Tensor, inc=1.0):
        """
        src, dst: LongTensor [B]
        t: FloatTensor [B] or scalar; assumes events are processed in nondecreasing time
        """
        # If t is per-edge, we update sequentially to be exact when a node repeats in batch.
        if t.dim() == 0:
            # scalar time for whole batch
            nodes = torch.unique(torch.cat([src, dst], dim=0))
            self._decay_to(nodes, t)
            self.k[src] += inc
            self.k[dst] += inc
            return

        # per-edge times: do sequential update
        B = src.numel()
        for i in range(B):
            ti = t[i]
            u = src[i].view(1)
            v = dst[i].view(1)
            self._decay_to(u, ti)
            self._decay_to(v, ti)
            self.k[u] += inc
            self.k[v] += inc

    def query(self, nodes: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """
        nodes: LongTensor [...]
        t: FloatTensor scalar or same shape broadcastable to nodes
        Returns k_u(t) for each node, without changing stored k/last_t.
        """
        last = self.last_t[nodes]
        dt = torch.clamp(t.to(last.dtype) - last, min=0.0)
        if self.beta > 0:
            return self.k[nodes] * torch.exp(-self.beta * dt)
        return self.k[nodes]

In [ ]:
import tgn.model.CommunityModel, importlib
importlib.reload(tgn.model.CommunityModel)

from tgn.model.CommunityModel import *
t0 = float(np.min(full_data.timestamps))
num_communities = 5
device = torch.device("mps")

model = CommunityModel(
    node_features=node_feat,      # np.ndarray [N, F]
    edge_features=edge_feat,      # np.ndarray [E, Fe] 你的 edge_idxs 指向这里
    device=device,
    embedding_dim=node_feat.shape[1],  # 也可以自己设更小，比如 64
    num_communities=num_communities,
    dropout=0.1,
    init_from_node_features=True,     # 没有 node features 就设 False（状态全 0）
    t0=t0,
).to(device)

model.reset_state(t0=t0)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)

In [76]:
import importlib, sys, time, my_loss
import tgn.utils.utils as utils
importlib.reload(my_loss)
importlib.reload(utils)
from tgn.utils.utils import *
from my_loss import * 
import numpy as np
import torch
import numpy as np
import torch

NUM_EPOCHS = 100
NUM_NEG = 30
lam = 1.0
beta_k = 1
t0 = float(np.min(full_data.timestamp_norm))

# ----- init neg sampler -----
all_nodes = np.unique(np.concatenate([full_data.sources, full_data.destinations])).astype(np.int64)
neg_sampler = NegativeNodeSampler(all_nodes, seed=0, use_seen_pool=True)

for epoch in range(NUM_EPOCHS):
    model.train()

    # 如果是 JODIE-style 有 state 表，建议每个 epoch 重置
    if hasattr(model, "reset_state"):
        model.reset_state(t0=t0)

    # 重置活跃度状态（避免跨 epoch 时间倒流）
    activity = ExpKernelActivity(num_nodes=num_nodes, beta=beta_k, device=device, t0=t0)

    # seen pool 重置可选
    neg_sampler.seen = set()

    for k in range(num_batches):
        start = BATCH_SIZE * k
        end = min(num_instance, BATCH_SIZE * (k + 1))

        sources_batch = full_data.sources[start:end]
        destinations_batch = full_data.destinations[start:end]
        timestamps_batch = full_data.timestamp_norm[start:end]
        edge_idxs_batch = full_data.edge_idxs[start:end]


        B = len(sources_batch)
        if B == 0:
            continue

        src_t = torch.as_tensor(sources_batch, dtype=torch.long, device=device)
        dst_t = torch.as_tensor(destinations_batch, dtype=torch.long, device=device)
        ts_t  = torch.as_tensor(timestamps_batch, dtype=torch.float32, device=device)


        activity.update_endpoints(src_t, dst_t, ts_t)
 
        # ---- sample negatives (node-only) ----
        neg_nodes_np = neg_sampler.sample(B, NUM_NEG)  # [B,R]
        neg_nodes_t  = torch.as_tensor(neg_nodes_np, dtype=torch.long, device=device)


        # ---- compute k BEFORE updating activity with this batch ----
        k_src = activity.query(src_t, ts_t)                    # [B]
        #print(k_src)
        k_neg = activity.query(neg_nodes_t, ts_t.unsqueeze(1)) # [B,R]

        optimizer.zero_grad()

        # ---- forward ----
        p_src, p_dst, p_neg = model.compute_community_prob(
            sources_batch, destinations_batch, neg_nodes_np, timestamps_batch, edge_idxs_batch
        )  # p_neg: [B,R,K]

        # ---- loss ----
        loss, dbg = temporal_modularity_k_weighted_neg_loss(
            p_src=p_src,
            p_dst=p_dst,
            p_neg=p_neg,
            k_src=k_src,
            k_neg=k_neg,
            lam=lam,
            symmetric=True,
            normalize_weights=True,
            collapse_from="all"
        )

        loss.backward()
        optimizer.step()


        """
        # ---- update model state from REAL events ----
        if hasattr(model, "update_states_from_events"):
            model.update_states_from_events(
                sources_batch, destinations_batch, timestamps_batch, edge_idxs_batch
            )
        """

        # ---- update activity + seen pool AFTER step ----
        activity.update_endpoints(src_t, dst_t, ts_t)
        neg_sampler.update_seen(sources_batch, destinations_batch)

        # ---- print dbg EVERY batch ----
        # 你也可以换成 logger.info(...)
        print(
            f"[Epoch {epoch+1}][Batch {k+1}/{num_batches}] "
            f"loss={float(loss.item()):.6f} "
            f"pos={dbg['pos_term']:.6f} neg={dbg['neg_term']:.6f} Q={dbg['Q']:.6f} "
            f"avg_k_src={dbg['avg_k_src']:.4f} avg_k_neg={dbg['avg_k_neg']:.4f}"
        )

[Epoch 1][Batch 1/1] loss=-0.160077 pos=0.773532 neg=0.607971 Q=0.165561 avg_k_src=1.1358 avg_k_neg=1.0759
[Epoch 2][Batch 1/1] loss=-0.171501 pos=0.769486 neg=0.592650 Q=0.176837 avg_k_src=1.1358 avg_k_neg=1.0718
[Epoch 3][Batch 1/1] loss=-0.163167 pos=0.757920 neg=0.589615 Q=0.168305 avg_k_src=1.1358 avg_k_neg=1.0731
[Epoch 4][Batch 1/1] loss=-0.147305 pos=0.734533 neg=0.582113 Q=0.152420 avg_k_src=1.1358 avg_k_neg=1.0789
[Epoch 5][Batch 1/1] loss=-0.166192 pos=0.778829 neg=0.607208 Q=0.171621 avg_k_src=1.1358 avg_k_neg=1.0741
[Epoch 6][Batch 1/1] loss=-0.152162 pos=0.784178 neg=0.626353 Q=0.157825 avg_k_src=1.1358 avg_k_neg=1.0709
[Epoch 7][Batch 1/1] loss=-0.158513 pos=0.774092 neg=0.610106 Q=0.163986 avg_k_src=1.1358 avg_k_neg=1.0739
[Epoch 8][Batch 1/1] loss=-0.174345 pos=0.769563 neg=0.590046 Q=0.179517 avg_k_src=1.1358 avg_k_neg=1.0728
[Epoch 9][Batch 1/1] loss=-0.152518 pos=0.740799 neg=0.582979 Q=0.157820 avg_k_src=1.1358 avg_k_neg=1.0734
[Epoch 10][Batch 1/1] loss=-0.153391 

In [75]:
import json
import numpy as np
import pandas as pd
import torch


def export_probs_to_csv_jodie(
    model,
    full_data,
    BATCH_SIZE: int,
    out_csv_path: str,
    id2node: dict,
    *,
    use_timestamp_norm_for_model: bool = True,
    t0: float | None = None,
):
    """
    Export p_src/p_dst (and argmax labels) for each interaction in chronological order.

    - Model time can use normalized timestamps (timestamp_norm) or raw timestamps.
    - CSV ALWAYS writes raw timestamps to column "timestamp".
    """
    model.eval()

    # --- timestamps for model vs for output ---
    ts_raw_all = full_data.timestamps  # ALWAYS export this
    ts_model_all = full_data.timestamp_norm if use_timestamp_norm_for_model else full_data.timestamps

    # ensure order is chronological w.r.t. model time
    order = np.argsort(ts_model_all, kind="mergesort")

    src_all = full_data.sources[order]
    dst_all = full_data.destinations[order]
    ts_model_all = ts_model_all[order]
    ts_raw_all = ts_raw_all[order]
    eidx_all = full_data.edge_idxs[order]

    # reset state
    if t0 is None:
        t0 = float(np.min(ts_model_all))
    if hasattr(model, "reset_state"):
        model.reset_state(t0=float(t0))

    rows = []

    def _map_id(x):
        x = int(x)
        return id2node[x] if id2node is not None else x

    with torch.no_grad():
        num_instance = len(src_all)
        num_batches = (num_instance + BATCH_SIZE - 1) // BATCH_SIZE

        for k in range(num_batches):
            start = BATCH_SIZE * k
            end = min(num_instance, BATCH_SIZE * (k + 1))

            sources_batch = src_all[start:end]
            destinations_batch = dst_all[start:end]
            ts_model_batch = ts_model_all[start:end]
            ts_raw_batch = ts_raw_all[start:end]
            edge_idxs_batch = eidx_all[start:end]

            B = len(sources_batch)
            if B == 0:
                continue

            # dummy negatives (export doesn't need true negatives)
            neg_nodes = destinations_batch

            p_src, p_dst, _ = model.compute_community_prob(
                sources_batch, destinations_batch, neg_nodes, ts_model_batch, edge_idxs_batch
            )

            p_src_np = p_src.detach().cpu().float().numpy()
            p_dst_np = p_dst.detach().cpu().float().numpy()
            src_commu = p_src_np.argmax(axis=1).astype(np.int64)
            dst_commu = p_dst_np.argmax(axis=1).astype(np.int64)

            for i in range(B):
                rows.append({
                    "source": _map_id(sources_batch[i]),
                    "destination": _map_id(destinations_batch[i]),
                    # ALWAYS write raw timestamp here
                    "timestamp": float(ts_raw_batch[i]),
                    "p_src": json.dumps(p_src_np[i].tolist()),
                    "p_dst": json.dumps(p_dst_np[i].tolist()),
                    "source_commu": int(src_commu[i]),
                    "destination_commu": int(dst_commu[i]),
                })

            # advance state with REAL events (use the SAME time scale as model used)
            if hasattr(model, "update_states_from_events"):
                model.update_states_from_events(
                    sources_batch, destinations_batch, ts_model_batch, edge_idxs_batch
                )

    df = pd.DataFrame(rows)
    df.to_csv(out_csv_path, index=False)
    print(f"Saved: {out_csv_path} (rows={len(df)})")
    return df

id2node = {idx: node for node, idx in node2id.items()}

export_probs_to_csv_jodie(
    model=model,
    full_data=full_data,
    BATCH_SIZE=BATCH_SIZE,
    out_csv_path="result/JODIE_community.csv",
    id2node=id2node,
    use_timestamp_norm_for_model=True,  # 模型内部仍用norm
    t0=float(np.min(full_data.timestamp_norm)),  # t0也建议用同一套model时间
).head(10)


Saved: result/JODIE_community.csv (rows=469)


,source,destination,timestamp,p_src,p_dst,source_commu,destination_commu
0,16,19,5.0,"[5.369382716224891e-09, 0.0, 0.0, 0.0, 1.0]","[4.3460871417858005e-16, 6.120451523947634e-19...",4,4
1,6,46,11.0,"[7.958715464440047e-09, 0.0, 0.0, 0.0, 1.0]","[2.4445904488515878e-14, 2.197903750912752e-34...",4,4
2,16,28,18.0,"[1.2328016829599164e-08, 0.0, 2.02362604245949...","[1.025487734553633e-17, 0.0, 0.0, 0.0, 1.0]",4,4
3,5,22,25.0,"[0.9997782111167908, 4.643037065932276e-09, 5....","[0.8264007568359375, 0.002390630077570677, 0.0...",0,0
4,8,35,31.0,"[0.9997504353523254, 3.2817222026724198e-22, 1...","[0.6216592192649841, 0.0016533946618437767, 0....",0,0
5,12,28,37.0,"[1.7722769463911225e-16, 0.0, 0.0, 0.0, 1.0]","[9.324719033293105e-17, 0.0, 0.0, 0.0, 1.0]",4,4
6,15,41,43.0,"[0.9992198944091797, 0.0, 0.0, 0.0, 0.00078008...","[0.9998588562011719, 0.0, 0.0, 0.0, 0.00014116...",0,0
7,7,49,52.0,"[0.9997832179069519, 1.399713350414179e-20, 1....","[0.9983983635902405, 0.0, 0.0, 0.0, 0.00160167...",0,0
8,38,43,60.0,"[0.9960787892341614, 0.00013328065688256174, 0...","[0.9997557997703552, 1.1956798819225981e-22, 6...",0,0
9,24,45,63.0,"[0.9756799340248108, 8.630101818301003e-25, 3....","[0.999855637550354, 0.0, 0.0, 0.0, 0.000144305...",0,0


In [38]:
import importlib, sys, time, my_loss
import tgn.utils.utils as utils
importlib.reload(my_loss)
importlib.reload(utils)
from tgn.utils.utils import *


NUM_EPOCHS = 5
NUM_NEG = 10
beta = 0.01

from my_loss import * 

all_nodes = np.concatenate([full_data.sources, full_data.destinations])
neg_sampler = RandOccurrenceSampler(seed=0, max_history=200000)

#origin_sampler = RandEdgeSampler(full_data.sources, full_data.destinations)


for epoch in range(NUM_EPOCHS):
    tgn = tgn.train()

    if USE_MEMORY:
        tgn.memory.__init_memory__()
    tgn.set_neighbor_finder(ngh_finder)

    # --------- epoch stats ---------
    epoch_loss_sum = 0.0
    epoch_pos_sum  = 0.0
    epoch_neg_sum  = 0.0
    epoch_Q_sum    = 0.0
    epoch_steps    = 0

    logger.info(f'Starting epoch {epoch+1} / {NUM_EPOCHS} ')
    for k in range(0, num_batches):
        start_idx = BATCH_SIZE * k
        end_idx = min(num_instance, BATCH_SIZE * (k + 1))

        sources_batch = full_data.sources[start_idx:end_idx]
        destinations_batch = full_data.destinations[start_idx:end_idx]
        timestamps_batch = full_data.timestamps[start_idx:end_idx]
        edge_idxs_batch = full_data.edge_idxs[start_idx:end_idx]

        src = torch.as_tensor(sources_batch, dtype=torch.long, device=device)
        dst = torch.as_tensor(destinations_batch, dtype=torch.long, device=device)
        ts  = torch.as_tensor(timestamps_batch, dtype=torch.float32, device=device)
        
        R = NUM_NEG
        B = len(sources_batch)

        # 1) 先从历史 occurrence 池采 (neg_node, neg_ts)
        neg_node_flat, neg_ts_flat = neg_sampler.sample_batch(
            ts_batch=timestamps_batch,
            R=R,
            strictly_past=True
        )

        # 2) 自适应得到 R_eff 并 reshape 成 [B,R_eff]
        L = neg_node_flat.size
        R_eff = L // B
        if R_eff == 0:
            neg_sampler.update(sources_batch, destinations_batch, timestamps_batch, include_src=True, include_dst=True)
            continue

        L_use = B * R_eff
        neg_nodes = neg_node_flat[:L_use].reshape(B, R_eff)   # [B,R_eff]
        neg_ts    = neg_ts_flat[:L_use].reshape(B, R_eff)     # [B,R_eff]


        optimizer.zero_grad()
        p_src, p_dst, p_neg = tgn.compute_community_prob(
            sources_batch, destinations_batch, neg_nodes, timestamps_batch, edge_idxs_batch
        )

        loss_mod, dbg = my_loss.temporal_modularity_coocc_weighted_loss(
            p_src=p_src,
            p_dst=p_dst,
            ts=ts,
            p_neg = p_neg,
            src=src,
            dst=dst,
            num_neg=NUM_NEG,
            beta=beta,
            lam=1.0,
        )
        
        loss_mod.backward()
        optimizer.step()

        neg_sampler.update(sources_batch, destinations_batch, timestamps_batch, include_src=True, include_dst=True)
        if USE_MEMORY:
            tgn.memory.detach_memory()

        epoch_loss_sum += float(loss_mod.item())
        epoch_pos_sum  += float(dbg.get("pos_term", 0.0))
        epoch_neg_sum  += float(dbg.get("neg_term", 0.0))
        epoch_Q_sum    += float(dbg.get("Q", 0.0))
        epoch_steps    += 1

    if epoch_steps > 0:
        avg_loss = epoch_loss_sum / epoch_steps
        avg_pos  = epoch_pos_sum  / epoch_steps
        avg_neg  = epoch_neg_sum  / epoch_steps
        avg_Q    = epoch_Q_sum    / epoch_steps
        logger.info(
            f"[Epoch {epoch+1}/{NUM_EPOCHS}] "
            f"avg_loss={avg_loss:.6f}  avg_pos={avg_pos:.6f}  avg_neg={avg_neg:.6f}  avg_Q={avg_Q:.6f}"
        )

    if USE_MEMORY:
        memory_backup = tgn.memory.backup_memory()
        tgn.memory.restore_memory(memory_backup)

INFO:root:Starting epoch 1 / 5 
INFO:root:Starting epoch 2 / 5 


RuntimeError: MPS backend out of memory (MPS allocated: 20.10 GiB, other allocations: 11.88 MiB, max allowed: 20.13 GiB). Tried to allocate 31.49 MiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

In [ ]:
import json
import numpy as np
import pandas as pd
import torch

def export_probs_to_csv(
    tgn,
    full_data,
    BATCH_SIZE: int,
    out_csv_path: str,
    id2node: dict
):
    tgn.eval()
    tgn.memory.__init_memory__()
    num_instance = len(full_data.sources)
    rows = []

    def _to_py_int(x):
        if isinstance(x, np.generic):
            return x.item()
        if torch.is_tensor(x):
            return x.item()
        return x

    def _map_id_to_node_name(x):
        x = _to_py_int(x)
        return id2node[int(x)]
    with torch.no_grad():
        for k in range((num_instance + BATCH_SIZE - 1) // BATCH_SIZE):
            start_idx = BATCH_SIZE * k
            end_idx = min(num_instance, BATCH_SIZE * (k + 1))

            sources_batch = full_data.sources[start_idx:end_idx]
            destinations_batch = full_data.destinations[start_idx:end_idx]
            timestamps_batch = full_data.timestamps[start_idx:end_idx]
            edge_idxs_batch = full_data.edge_idxs[start_idx:end_idx]
            
            p_src, p_dst = tgn.compute_community_prob(
                sources_batch, destinations_batch, neg_node_batch, timestamps_batch, edge_idxs_batch
            )

            p_src_np = p_src.detach().cpu().float().numpy()
            p_dst_np = p_dst.detach().cpu().float().numpy()

            src_commu = p_src_np.argmax(axis=1).astype(np.int64)
            dst_commu = p_dst_np.argmax(axis=1).astype(np.int64)

            for i in range(end_idx - start_idx):
                rows.append({
                    "source": _map_id_to_node_name(sources_batch[i]),
                    "destination": _map_id_to_node_name(destinations_batch[i]),
                    "timestamp": float(_to_py_int(timestamps_batch[i])),
                    "p_src": json.dumps(p_src_np[i].tolist()),
                    "p_dst": json.dumps(p_dst_np[i].tolist()),
                    "source_commu": int(src_commu[i]),
                    "destination_commu": int(dst_commu[i]),
                })

    df = pd.DataFrame(rows, columns=[
        "source", "destination", "timestamp",
        "p_src", "p_dst", "source_commu", "destination_commu"
    ])
    df.to_csv(out_csv_path, index=False)
    print(f"Saved: {out_csv_path}  (rows={len(df)})")
id2node = {idx: node for node, idx in node2id.items()}


export_probs_to_csv(tgn, full_data, BATCH_SIZE, "result/TGN_community.csv", id2node=id2node)

TypeError: TGN.compute_community_prob() missing 1 required positional argument: 'edge_idxs'

In [70]:
df = pd.read_csv("result/JODIE_community.csv")
print("unique source_commu:", df["source_commu"].nunique())
print("unique destination_commu:", df["destination_commu"].nunique())
print(df["source_commu"].value_counts().head(10))

unique source_commu: 3
unique destination_commu: 3
source_commu
1    239
4    177
3     53
Name: count, dtype: int64
